# Multi-Head Latent Attention (MLA)

Multi-Head Latent Attention is the attention variant used in DeepSeek. Its goal is to
shrink the **KV cache**, the memory a transformer keeps for every past token during
generation.

Standard attention caches the full keys and values, costing `2 * d_model` numbers per
token. MLA instead **compresses** each token into a small latent vector of size
`kv_latent_dim` and caches only that. Keys and values are reconstructed from the latent
when they are needed:

$$
c_{kv} = \text{LayerNorm}(W_{dkv}\,x), \qquad K = W_{uk}\,c_{kv}, \qquad V = W_{uv}\,c_{kv}
$$

This version is "ropeless" (no rotary position embedding) so the core idea stays clear. It
also uses the **weight absorption** trick: since the scores are `q (W_uk c_{kv})^T`, the
matrices `W_q` and `W_uk` can be multiplied together once into `absorbed_k`, so keys never
need to be expanded to full width when computing scores.

In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

## The layer

The projections are:

| Layer | Maps | Purpose |
| - | - | - |
| `W_dkv` | `d_model -> kv_latent_dim` | compress a token into the KV latent |
| `W_uk` | `kv_latent_dim -> d_model` | rebuild keys from the latent |
| `W_uv` | `kv_latent_dim -> d_model` | rebuild values from the latent |
| `W_q` | `d_model -> d_model` | queries (folded into `absorbed_k`) |
| `W_o` | `d_model -> d_model` | output projection |

`forward` returns both the attention output and the updated latent cache `c_kv`, so the
cache can be reused for the next token during generation.

In [2]:
class RopelessMLA(nn.Module):
    """Multi-Head Latent Attention (without RoPE).

    Instead of caching full keys and values (2 * d_model numbers per token), MLA
    compresses each token into a small latent vector of size `kv_latent_dim` and caches
    only that. Keys and values are reconstructed from the latent when needed, and the
    query/key-up projections are 'absorbed' into one matrix so keys never have to be
    expanded to full width at score time.
    """
    def __init__(self, d_model, n_heads, kv_latent_dim):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads          # dimension per head

        # Projection layers
        self.W_q   = nn.Linear(d_model, d_model, bias=False)         # query projection
        self.W_dkv = nn.Linear(d_model, kv_latent_dim, bias=False)   # down: x -> latent
        self.W_uk  = nn.Linear(kv_latent_dim, d_model, bias=False)   # up: latent -> keys
        self.W_uv  = nn.Linear(kv_latent_dim, d_model, bias=False)   # up: latent -> values
        self.W_o   = nn.Linear(d_model, d_model, bias=False)         # output projection

        self.ln = nn.LayerNorm(kv_latent_dim)
        self.register_buffer('absorbed_k', None)     # cached W_q @ W_uk

    def forward(self, x, kv_cache=None, past_length=0):
        B, S, D = x.size()

        # Absorb query and key-up projections once: (D, D) @ (D, latent) -> (D, latent)
        if self.absorbed_k is None:
            absorbed = torch.matmul(self.W_q.weight, self.W_uk.weight)
            self.absorbed_k = absorbed.view(self.n_heads, self.head_dim, -1)

        # Compress x into the KV latent, then append to the cache (if any)
        new_c_kv = self.ln(self.W_dkv(x))                    # (B, S, latent)
        if kv_cache is None:
            c_kv = new_c_kv
        else:
            c_kv = torch.cat([kv_cache, new_c_kv], dim=1)    # (B, S_full, latent)
        S_full = c_kv.size(1)

        # Decompress values to full width and split into heads
        v = self.W_uv(c_kv)                                  # (B, S_full, D)
        v = v.view(B, S_full, self.n_heads, self.head_dim).transpose(1, 2)  # (B, H, S_full, head_dim)

        # Query comes straight from x, because W_q is folded into absorbed_k
        q = x.view(B, S, self.n_heads, self.head_dim)        # (B, S, H, head_dim)

        # Attention scores per head (keys stay in latent space via absorbed_k)
        atten_scores = torch.zeros(B, self.n_heads, S, S_full, device=x.device)
        for h in range(self.n_heads):
            tmp = torch.matmul(q[:, :, h], self.absorbed_k[h])        # (B, S, latent)
            atten_scores[:, h] = torch.bmm(tmp, c_kv.transpose(1, 2)) # (B, S, S_full)

        # Scale
        atten_scores = atten_scores / (self.head_dim ** 0.5)

        # Causal mask: query i may attend to key j where j <= past_length + i
        mask = torch.tril(torch.ones(S, S_full, device=x.device), diagonal=past_length)
        atten_scores = atten_scores.masked_fill(mask.unsqueeze(0).unsqueeze(0) == 0, float('-inf'))

        # Attention weights
        atten_weight = torch.softmax(atten_scores, dim=-1)

        # Weighted sum of values, per head, then merge heads
        out = torch.zeros(B, self.n_heads, S, self.head_dim, device=x.device)
        for h in range(self.n_heads):
            out[:, h] = torch.bmm(atten_weight[:, h], v[:, h])        # (B, S, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, S, D)          # (B, S, D)

        # Output projection; also return the updated latent cache
        return self.W_o(out), c_kv

## Running the layer

A forward pass on one sequence of five tokens. The output keeps the `d_model` width, while
the cache stores only `kv_latent_dim` numbers per token.

In [3]:
torch.manual_seed(0)
mla = RopelessMLA(d_model=8, n_heads=2, kv_latent_dim=4).eval()

x = torch.randn(1, 5, 8)          # (batch=1, seq_len=5, d_model=8)
with torch.no_grad():
    output, c_kv = mla(x)

print("input shape        :", x.shape)
print("output shape       :", output.shape)     # (1, 5, 8)
print("latent cache shape :", c_kv.shape)       # (1, 5, 4)  only kv_latent_dim per token

input shape        : torch.Size([1, 5, 8])
output shape       : torch.Size([1, 5, 8])
latent cache shape : torch.Size([1, 5, 4])


## The payoff: a smaller KV cache

Generating one token at a time and caching only the latent must give the same result as
running the whole sequence at once. The cell below verifies that, then compares the cache
size against a standard key and value cache.

In [4]:
# The point of MLA: generate token by token, caching ONLY the small latent c_kv.
torch.manual_seed(0)
with torch.no_grad():
    cache, outputs = None, []
    for t in range(x.size(1)):
        step_out, cache = mla(x[:, t:t+1, :], kv_cache=cache, past_length=t)
        outputs.append(step_out)
    incremental = torch.cat(outputs, dim=1)

# Incremental decoding must match the single-shot full forward pass
print("max difference vs full forward:", (output - incremental).abs().max().item())

# Memory compared with a standard key/value cache
d_model, kv_latent_dim = 8, 4
standard  = 2 * d_model        # standard attention caches K AND V, full width
mla_cache = kv_latent_dim      # MLA caches one small latent
print(f"standard KV cache : {standard} floats / token")
print(f"MLA latent cache  : {mla_cache} floats / token")
print(f"compression       : {standard / mla_cache:.1f}x smaller")

max difference vs full forward: 7.450580596923828e-08
standard KV cache : 16 floats / token
MLA latent cache  : 4 floats / token
compression       : 4.0x smaller
